In [1]:
from clustering_variable_label_dictionary import value_labels, display_names
import pandas as pd

In [2]:
def recode_nssec(df):

    df = df.copy()

    df.loc[df["NSSEC5"].isna() & (df["Age9"] >= 8), "NSSEC5"] = 5

    return df

In [3]:
def recode_workstat(df):
    df = df.copy()

    mapping = {1: 1,
               2: 2,
               3: 3,
               4: 3,
               5: 4,
               6: 5,
               7: 6,
               8: 7,
               9: 7,
               10: 8}

    df["WorkStat8"] = df["WorkStat10"].map(mapping)

    return df

In [4]:
def recode_hhliv(df):
    df = df.copy()

    mapping = {1: 1,
               2: 2,
               3: 3,
               4: 3,
               5: 4,
               6: 5,
               7: 6,
               8: 7,
               9: 7,
               10: 8,
               11: 9,
               12: 9}

    df["HHLiv9"] = df["HHLiv12"].map(mapping)

    return df

In [5]:
overall_df = pd.read_csv("../../data/master_data/2016_to_2023_full_preprocessed_data_set.csv.gz")

In [6]:
overall_df = recode_nssec(overall_df)
overall_df = recode_workstat(overall_df)
overall_df = recode_hhliv(overall_df)

In [7]:
cluster_cols = ["serial",
                "year",
                "Age9",
                "Gend3",
                "Eth7",
                "Disab2_POP",
                "Educ6",
                "NSSEC5",
                "IMD10",
                "WorkStat8",
                "Child4",
                "HHLiv9",
                "Motiva_POP",
                "motivd_POP"]

In [8]:
cluster_vars = [col for col in cluster_cols if col not in ["serial", "year"]]

cluster_df = overall_df[cluster_cols].copy()

In [9]:
sensitivity_df = cluster_df.copy()

sensitivity_df["n_missing"] = (sensitivity_df[cluster_vars].isna().sum(axis=1))

sensitivity_df["excluded"] = sensitivity_df["n_missing"] > 0

In [10]:
def exclusion_by_group(df, variable):

    temp = df[df[variable].notna()]

    summary = (temp.groupby(variable)["excluded"].agg(N="size", N_excluded="sum", Percent_excluded="mean").reset_index())

    summary["Percent_excluded"] = summary["Percent_excluded"] * 100

    return summary

In [11]:
summary_rows = []

for var in cluster_vars:

    n_missing = sensitivity_df[var].isna().sum()
    percent_missing = sensitivity_df[var].isna().mean() * 100

    group_summary = exclusion_by_group(sensitivity_df, var)

    highest = group_summary.loc[group_summary["Percent_excluded"].idxmax()]

    category_code = int(highest[var])

    category_label_code = (category_code - int(cluster_df[var].min()))

    category_label = value_labels[var][category_label_code]

    summary_rows.append({"Variable": display_names[var],
                         "N missing": n_missing,
                         "% missing": percent_missing,
                         "Highest-exclusion category": category_label,
                         "Category N": highest["N"],
                         "% excluded": highest["Percent_excluded"]})

sensitivity_summary = pd.DataFrame(summary_rows)

sensitivity_summary["N missing"] = (sensitivity_summary["N missing"].astype(int))
sensitivity_summary["Category N"] = (sensitivity_summary["Category N"].astype(int))
sensitivity_summary["% missing"] = (sensitivity_summary["% missing"].round(2))
sensitivity_summary["% excluded"] = (sensitivity_summary["% excluded"].round(2))

display(sensitivity_summary)

,Variable,N missing,% missing,Highest-exclusion category,Category N,% excluded
0,Age,1127,0.96,85+,1712,53.86
1,Gender,259,0.22,Other,475,61.26
2,Ethnicity,8075,6.86,Black,7399,29.04
3,Disability,7944,6.75,Disability,16272,30.65
4,Education,8016,6.81,No qualifications,6420,42.73
5,Socio-economic Status,172,0.15,NS SEC 9: Students and other / unclassified,10807,46.21
6,Deprivation,0,0.00,Most deprived decile,7990,33.38
7,Employment Status,2863,2.43,Other,4030,41.09
8,Children,1878,1.60,3 or more children,4174,33.61
9,Household Composition,9705,8.25,Other/complex household,5350,37.29


In [12]:
sensitivity_df.loc[sensitivity_df["Age9"] == 9, cluster_vars].isna().mean().mul(100).sort_values(ascending=False)

motivd_POP    33.820093
Motiva_POP    25.116822
Educ6         17.114486
HHLiv9        10.105140
Disab2_POP     9.112150
WorkStat8      8.878505
Eth7           7.242991
Child4         4.672897
Gend3          0.408879
Age9           0.000000
IMD10          0.000000
NSSEC5         0.000000
dtype: float64

In [13]:
sensitivity_df.loc[sensitivity_df["Gend3"] == 3, cluster_vars].isna().mean().mul(100).sort_values(ascending=False)

Disab2_POP    41.263158
Eth7          41.052632
motivd_POP    14.526316
HHLiv9        11.789474
Motiva_POP    11.578947
Educ6          8.000000
Age9           5.263158
WorkStat8      4.210526
Child4         2.105263
NSSEC5         0.421053
Gend3          0.000000
IMD10          0.000000
dtype: float64

In [14]:
cluster_df = cluster_df.dropna().copy()

In [15]:
cluster_df[cluster_vars] = cluster_df[cluster_vars].astype(int)

for col in cluster_vars:
    cluster_df[col] -= cluster_df[col].min()

In [16]:
# cluster_df.to_csv("../../data/master_data/2016_to_2023_clustering_input_data.csv", index=False)